# 34 — Error analysis y diagnósticos por segmento

Analiza el modelo ya seleccionado exclusivamente sobre la mitad de VALIDATION reservada para selección. No cambia el modelo y no consume TEST.

In [1]:
from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec
from argos.config.modeling import load_modeling_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()

m3_gate = paths.docs / "decisions" / "M3_GATE.md"
assert m3_gate.exists() and "M3 ✅ CLOSED" in m3_gate.read_text(encoding="utf-8"), (
    "M3 no está formalmente cerrado. No iniciar Modeling."
)
GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")
MODEL_SPEC = load_modeling_spec(paths.configs / "modeling.toml")
MODEL_DIR = paths.root / "artifacts" / "models" / "m4"
MANIFEST_DIR = paths.root / "artifacts" / "manifests"

{
    "gold_schema": GOLD_SPEC.schema_version,
    "modeling_schema": MODEL_SPEC.schema_version,
    "target": MODEL_SPEC.target,
    "model_matrix": MODEL_SPEC.model_matrix,
    "operational_review_fraction": MODEL_SPEC.ranking.operational_fraction,
}

assert (MODEL_DIR / "selected_model.json").exists(), "Ejecuta 33 primero."
from argos.models.error_analysis import BUILD_IMPLEMENTATION
BUILD_IMPLEMENTATION


'm4-34-validation-error-analysis-v1'

In [2]:
from argos.models.error_analysis import build_error_analysis

result = build_error_analysis(
    root=paths.root,
    modeling_spec=MODEL_SPEC,
    reports_root=paths.reports,
    manifest_dir=MANIFEST_DIR,
)
result.segments.head(50)


,segment,value,rows,positives,prevalence,mean_score,flagged_at_operational_k,tp,fp,fn,precision_at_global_cutoff,recall_at_global_cutoff
0,t0_quarter,3,230322,634,0.002753,0.006003,2652,139,2513,495,0.052413,0.219243
1,t0_quarter,4,197754,934,0.004723,0.006002,2004,153,1851,781,0.076347,0.163812
2,departamento,Antioquia,43371,196,0.004519,0.006343,180,15,165,181,0.083333,0.076531
3,departamento,Arauca,2052,11,0.005361,0.006250,15,1,14,10,0.066667,0.090909
4,departamento,Atlántico,15455,39,0.002523,0.005082,111,3,108,36,0.027027,0.076923
5,departamento,Bolívar,13856,88,0.006351,0.005591,71,7,64,81,0.098592,0.079545
6,departamento,Boyacá,14653,128,0.008735,0.011911,713,22,691,106,0.030856,0.171875
7,departamento,Caldas,8195,56,0.006833,0.007527,118,9,109,47,0.076271,0.160714
8,departamento,Caquetá,2602,68,0.026134,0.005228,13,3,10,65,0.230769,0.044118
9,departamento,Casanare,8664,26,0.003001,0.004790,25,3,22,23,0.120000,0.115385


In [3]:
result.score_deciles

,score_decile,rows,positives,mean_score,min_score,max_score,prevalence
0,1,42808,1052,0.032206,7.437287e-03,0.130938,0.024575
1,2,42808,198,0.005470,4.545455e-03,0.007437,0.004625
2,3,42807,104,0.003393,2.935995e-03,0.004545,0.002430
3,4,42808,58,0.002845,2.738059e-03,0.002936,0.001355
4,5,42807,26,0.002725,2.712016e-03,0.002738,0.000607
5,6,42808,27,0.002712,2.712016e-03,0.002712,0.000631
6,7,42807,29,0.002712,2.712016e-03,0.002712,0.000677
7,8,42808,25,0.002712,2.712016e-03,0.002712,0.000584
8,9,42807,22,0.002712,2.712016e-03,0.002712,0.000514
9,10,42808,27,0.002539,1.000000e-07,0.002712,0.000631


In [4]:
result.validation

,check_id,category,blocking,passed,observed,requirement
0,error_analysis_selection_population,evaluation,True,True,"{""joined_rows"": 428076, ""predictions"": 428076}",error analysis preserves the frozen validation...
1,error_analysis_segments_available,diagnostics,True,True,"{""segment_rows"": 43, ""segments"": [""departament...",error analysis reports sufficiently supported ...
2,error_analysis_test_isolation,test_isolation,True,True,"{""consumed_split"": ""validation_selection""}",notebook 34 does not consume TEST rows


In [5]:
blocking = result.validation.loc[result.validation["blocking"]]
print("34 ✅ PASS" if result.ready_for_next_notebook else "34 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
assert result.ready_for_next_notebook


34 ✅ PASS
Blocking checks: 3/3
